In [ ]:
# ============================================================================
# Imports
# ============================================================================

import networkx as nx
import numpy as np
import pandas as pd

from hermes.config import GRAPH_DIR
from hermes.graph.io import load_graphml

In [ ]:
# ============================================================================
# Load territorial graph
# ============================================================================

graph = load_graphml(
    GRAPH_DIR / "municipality.graphml"
)

print(
    f"Nodes: {graph.number_of_nodes():,}"
)
print(
    f"Edges: {graph.number_of_edges():,}"
)

In [ ]:
# ============================================================================
# Compute weighted node strengths
# ============================================================================

out_strength = dict(
    graph.out_degree(
        weight="commuters",
    )
)

in_strength = dict(
    graph.in_degree(
        weight="commuters",
    )
)

network_metrics = pd.DataFrame(
    {
        "insee_code": list(graph.nodes),
        "out_strength": [
            out_strength[node]
            for node in graph.nodes
        ],
        "in_strength": [
            in_strength[node]
            for node in graph.nodes
        ],
    }
)

network_metrics.head()

In [ ]:
# ============================================================================
# Add municipality attributes
# ============================================================================

node_attributes = pd.DataFrame.from_dict(
    dict(graph.nodes(data=True)),
    orient="index",
)

node_attributes.index.name = "insee_code"

node_attributes = (
    node_attributes
    .reset_index()
    [
        [
            "insee_code",
            "municipality",
            "entity_type",
            "department_code",
            "region_code",
            "population",
            "employed",
            "total_jobs",
        ]
    ]
)

network_metrics = network_metrics.merge(
    node_attributes,
    on="insee_code",
    how="left",
)

network_metrics.head()

In [ ]:
# ============================================================================
# Compute unweighted node degrees
# ============================================================================

network_metrics["out_degree"] = (
    network_metrics["insee_code"]
    .map(
        dict(
            graph.out_degree()
        )
    )
)

network_metrics["in_degree"] = (
    network_metrics["insee_code"]
    .map(
        dict(
            graph.in_degree()
        )
    )
)

network_metrics[
    [
        "municipality",
        "out_degree",
        "in_degree",
        "out_strength",
        "in_strength",
    ]
].head()

In [ ]:
network_metrics[
    [
        "out_degree",
        "in_degree",
        "out_strength",
        "in_strength",
    ]
].describe()

In [ ]:
# ============================================================================
# Identify municipalities with highest in-degree
# ============================================================================

top_in_degree = (
    network_metrics
    .sort_values(
        "in_degree",
        ascending=False,
    )
    [
        [
            "insee_code",
            "municipality",
            "entity_type",
            "department_code",
            "in_degree",
            "in_strength",
        ]
    ]
    .head(20)
)

top_in_degree

In [ ]:
# ============================================================================
# Identify municipalities with highest out-degree
# ============================================================================

top_out_degree = (
    network_metrics
    .sort_values(
        "out_degree",
        ascending=False,
    )
    [
        [
            "insee_code",
            "municipality",
            "entity_type",
            "department_code",
            "out_degree",
            "out_strength",
        ]
    ]
    .head(20)
)

top_out_degree

In [ ]:
# ============================================================================
# Compute commuting balance
# ============================================================================

network_metrics["commuting_balance"] = (
    network_metrics["in_strength"]
    - network_metrics["out_strength"]
)

network_metrics["commuting_ratio"] = (
    network_metrics["in_strength"]
    / network_metrics["out_strength"].replace(0, pd.NA)
)

network_metrics[
    [
        "municipality",
        "entity_type",
        "in_strength",
        "out_strength",
        "commuting_balance",
        "commuting_ratio",
    ]
].describe()

In [ ]:
# ============================================================================
# Identify strongest employment attractors and residential emitters
# ============================================================================

top_attractors = (
    network_metrics
    .nlargest(20, "commuting_balance")
    [
        [
            "insee_code",
            "municipality",
            "entity_type",
            "department_code",
            "in_strength",
            "out_strength",
            "commuting_balance",
        ]
    ]
)

top_emitters = (
    network_metrics
    .nsmallest(20, "commuting_balance")
    [
        [
            "insee_code",
            "municipality",
            "entity_type",
            "department_code",
            "in_strength",
            "out_strength",
            "commuting_balance",
        ]
    ]
)

display(top_attractors)
display(top_emitters)

In [ ]:
# ============================================================================
# Visualize commuting inflows and outflows
# ============================================================================

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(
    network_metrics["out_strength"],
    network_metrics["in_strength"],
    alpha=0.35,
    s=18,
)

max_strength = max(
    network_metrics["out_strength"].max(),
    network_metrics["in_strength"].max(),
)

ax.plot(
    [0, max_strength],
    [0, max_strength],
    linestyle="--",
    linewidth=1,
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Outgoing commuters")
ax.set_ylabel("Incoming commuters")
ax.set_title("Commuting inflows vs outflows")

plt.savefig(
    "../assets/commuting_inflows_vs_outflows.png",
    dpi=300,
    bbox_inches="tight",
)

plt.tight_layout()
plt.show()

## Mobility features relevant to (e-)bike adoption

This section derives interpretable commuting indicators that may help explain
territorial differences in e-bike adoption.

In [ ]:
# ============================================================================
# Compute local retention and commuting dependency
# ============================================================================

self_loops = {
    node: (
        graph[node][node]["commuters"]
        if graph.has_edge(node, node)
        else 0.0
    )
    for node in graph.nodes
}

network_metrics["internal_flow"] = (
    network_metrics["insee_code"]
    .map(self_loops)
)

network_metrics["external_outflow"] = (
    network_metrics["out_strength"]
    - network_metrics["internal_flow"]
)

network_metrics["retention_rate"] = (
    network_metrics["internal_flow"]
    / network_metrics["out_strength"].replace(0, np.nan)
)

network_metrics["external_commuting_rate"] = (
    network_metrics["external_outflow"]
    / network_metrics["out_strength"].replace(0, np.nan)
)

network_metrics[
    [
        "municipality",
        "internal_flow",
        "external_outflow",
        "retention_rate",
        "external_commuting_rate",
    ]
].describe()

In [ ]:
# ============================================================================
# Visualize external commuting dependency
# ============================================================================

fig, ax = plt.subplots(figsize=(8, 5))

network_metrics["external_commuting_rate"].dropna().hist(
    bins=40,
    ax=ax,
)

ax.set_xlabel("External commuting rate")
ax.set_ylabel("Number of municipalities")
ax.set_title("Distribution of external commuting dependency")

plt.tight_layout()

fig.savefig(

    "../assets/figures/external_commuting_rate_distribution.png",

    dpi=300,

    bbox_inches="tight",
)

plt.show()

<span style="color:green">
<b>RESULT</b>: in a typical municipality, 80 % of employed work outside their residential municipality.
</span>

In [ ]:
# ============================================================================
# Compute commuting distances
# ============================================================================

from math import radians, sin, cos, sqrt, atan2


def haversine_distance(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float,
) -> float:
    """
    Compute great-circle distance between two coordinates in kilometres.
    """

    earth_radius_km = 6371.0

    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)

    a = (
        sin(dlat / 2) ** 2
        + cos(radians(lat1))
        * cos(radians(lat2))
        * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1 - a),
    )

    return earth_radius_km * c

In [ ]:
# ============================================================================
# Compute weighted mean commuting distance
# ============================================================================

distance_records = []

for origin in graph.nodes:

    origin_lat = graph.nodes[origin].get("latitude")
    origin_lon = graph.nodes[origin].get("longitude")

    if pd.isna(origin_lat) or pd.isna(origin_lon):
        continue

    weighted_distance_sum = 0.0
    commuter_sum = 0.0

    for _, destination, attributes in graph.out_edges(
        origin,
        data=True,
    ):

        if origin == destination:
            continue

        destination_lat = graph.nodes[destination].get("latitude")
        destination_lon = graph.nodes[destination].get("longitude")

        if (
            pd.isna(destination_lat)
            or pd.isna(destination_lon)
        ):
            continue

        commuters = attributes.get(
            "commuters",
            0.0,
        )

        distance_km = haversine_distance(
            origin_lat,
            origin_lon,
            destination_lat,
            destination_lon,
        )

        weighted_distance_sum += (
            distance_km * commuters
        )

        commuter_sum += commuters

    mean_distance = (
        weighted_distance_sum / commuter_sum
        if commuter_sum > 0
        else np.nan
    )

    distance_records.append(
        {
            "insee_code": origin,
            "mean_external_commuting_distance_km": mean_distance,
        }
    )

distance_metrics = pd.DataFrame(
    distance_records
)

network_metrics = network_metrics.merge(
    distance_metrics,
    on="insee_code",
    how="left",
)

network_metrics[
    "mean_external_commuting_distance_km"
].describe()

In [ ]:
# ============================================================================
# Plot mean external commuting distance distribution
# ============================================================================

distance_data = network_metrics[
    "mean_external_commuting_distance_km"
].dropna()

fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(
    distance_data[distance_data <= 100],
    bins=50,
)

ax.axvline(
    distance_data.median(),
    linestyle="--",
    label=f"Median: {distance_data.median():.1f} km",
)

ax.set_title(
    "Distribution of Mean External Commuting Distance"
)
ax.set_xlabel("Mean commuting distance (km)")
ax.set_ylabel("Number of municipalities")
ax.legend()

fig.tight_layout()

fig.savefig(
    "../assets/figures/mean_external_commuting_distance.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================================
# Compute distance for each external commuting flow
# ============================================================================

flow_distances = []

for origin, destination, attributes in graph.edges(data=True):

    if origin == destination:
        continue

    origin_lat = graph.nodes[origin].get("latitude")
    origin_lon = graph.nodes[origin].get("longitude")

    destination_lat = graph.nodes[destination].get("latitude")
    destination_lon = graph.nodes[destination].get("longitude")

    if (
        pd.isna(origin_lat)
        or pd.isna(origin_lon)
        or pd.isna(destination_lat)
        or pd.isna(destination_lon)
    ):
        continue

    distance_km = haversine_distance(
        origin_lat,
        origin_lon,
        destination_lat,
        destination_lon,
    )

    flow_distances.append(
        {
            "origin_insee_code": origin,
            "destination_insee_code": destination,
            "commuters": attributes["commuters"],
            "distance_km": distance_km,
        }
    )

flow_distances = pd.DataFrame(flow_distances)

flow_distances.head()

In [ ]:
# ============================================================================
# Classify external commuting flows by distance
# ============================================================================

distance_bins = [
    0,
    5,
    10,
    20,
    40,
    np.inf,
]

distance_labels = [
    "0_5_km",
    "5_10_km",
    "10_20_km",
    "20_40_km",
    "over_40_km",
]

flow_distances["distance_band"] = pd.cut(
    flow_distances["distance_km"],
    bins=distance_bins,
    labels=distance_labels,
    right=False,
)

flow_distances[
    [
        "origin_insee_code",
        "destination_insee_code",
        "commuters",
        "distance_km",
        "distance_band",
    ]
].head(20)

In [ ]:
# ============================================================================
# Aggregate external commuting flows by distance band
# ============================================================================

distance_band_flows = (
    flow_distances
    .groupby(
        [
            "origin_insee_code",
            "distance_band",
        ],
        observed=True,
    )["commuters"]
    .sum()
    .unstack(
        fill_value=0,
    )
    .reset_index()
)

distance_band_flows = distance_band_flows.rename(
    columns={
        "0_5_km": "external_commuters_0_5km",
        "5_10_km": "external_commuters_5_10km",
        "10_20_km": "external_commuters_10_20km",
        "20_40_km": "external_commuters_20_40km",
        "over_40_km": "external_commuters_over_40km",
    }
)

distance_band_flows.head()

In [ ]:
# ============================================================================
# Compute shares of external commuters by distance band
# ============================================================================

distance_share_columns = [
    "external_commuters_0_5km",
    "external_commuters_5_10km",
    "external_commuters_10_20km",
    "external_commuters_20_40km",
    "external_commuters_over_40km",
]

distance_band_flows["external_commuters_total"] = (
    distance_band_flows[distance_share_columns]
    .sum(axis=1)
)

for column in distance_share_columns:

    share_column = column.replace(
        "external_commuters_",
        "share_external_commuters_",
    )

    distance_band_flows[share_column] = (
        distance_band_flows[column]
        / distance_band_flows["external_commuters_total"].replace(
            0,
            np.nan,
        )
    )

distance_band_flows[
    [
        "origin_insee_code",
        "share_external_commuters_0_5km",
        "share_external_commuters_5_10km",
        "share_external_commuters_10_20km",
        "share_external_commuters_20_40km",
        "share_external_commuters_over_40km",
    ]
].head()

In [ ]:
# ============================================================================
# Validate distance-band shares
# ============================================================================

share_columns = [
    "share_external_commuters_0_5km",
    "share_external_commuters_5_10km",
    "share_external_commuters_10_20km",
    "share_external_commuters_20_40km",
    "share_external_commuters_over_40km",
]

distance_band_flows[
    share_columns
].sum(axis=1).describe()

In [ ]:
# ============================================================================
# Merge commuting distance shares into network metrics
# ============================================================================

distance_share_metrics = distance_band_flows[
    [
        "origin_insee_code",
        "share_external_commuters_0_5km",
        "share_external_commuters_5_10km",
        "share_external_commuters_10_20km",
        "share_external_commuters_20_40km",
        "share_external_commuters_over_40km",
    ]
].rename(
    columns={
        "origin_insee_code": "insee_code",
    }
)

network_metrics = network_metrics.merge(
    distance_share_metrics,
    on="insee_code",
    how="left",
)

network_metrics[
    [
        "municipality",
        "retention_rate",
        "external_commuting_rate",
        "mean_external_commuting_distance_km",
        "share_external_commuters_0_5km",
        "share_external_commuters_5_10km",
        "share_external_commuters_10_20km",
        "share_external_commuters_20_40km",
        "share_external_commuters_over_40km",
    ]
].head()

In [ ]:
# ============================================================================
# Compute bike-relevant external commuting distance features
# ============================================================================

network_metrics["share_external_commuters_under_20km"] = (
    network_metrics["share_external_commuters_0_5km"]
    + network_metrics["share_external_commuters_5_10km"]
    + network_metrics["share_external_commuters_10_20km"]
)

network_metrics["share_external_commuters_5_20km"] = (
    network_metrics["share_external_commuters_5_10km"]
    + network_metrics["share_external_commuters_10_20km"]
)

network_metrics[
    [
        "share_external_commuters_under_20km",
        "share_external_commuters_5_20km",
    ]
].describe()

<span style="color:green">
<b>RESULTS:</b> ~70 % of external commuters stay work within a 20km radius, ~58 % within a the range 5-20km.
</span>

In [ ]:
# ============================================================================
# Inspect Villefranche-sur-Saône mobility profile
# ============================================================================

villefranche_profile = network_metrics[
    network_metrics["municipality"]
    .str.contains(
        "Villefranche-sur-Saône",
        case=False,
        na=False,
    )
]

villefranche_profile[
    [
        "insee_code",
        "municipality",
        "population",
        "employed",
        "total_jobs",
        "retention_rate",
        "external_commuting_rate",
        "mean_external_commuting_distance_km",
        "share_external_commuters_0_5km",
        "share_external_commuters_5_10km",
        "share_external_commuters_10_20km",
        "share_external_commuters_20_40km",
        "share_external_commuters_over_40km",
        "share_external_commuters_under_20km",
        "share_external_commuters_5_20km",
    ]
].T

<span style="color:green">
<b>RESULTS:</b>:  

In Villefranche-sur-Saône, 59.95 % of workers commute outside the municipality, 40.05% within the municipality.

66.8% of workers commute within a 20 km radius, 42.3% of them within the 5-20km range.

The mean commuting distance is 37.2 km, however 2/3 of external commuters work within the 20 km radius.

We might consider three distinct mobility markets:
- intra-municipality: ~40 % of workers, for whom both bikes and e-bikes are relevant. Topography, age, biking infrastructures, etc. will affect atractiveness of bike commuting.
- short inter-municipality: 42.3% of external commuters work within a distance of 5-20km. This segment might be interested in trading cars for bikes.
- long-distance: 25.9 % of commuter work within a 20-40 km distance. This segment might be interested in using intermodality including bikes and folding bikes.
</span>

In [ ]:
# ============================================================================
# Position Villefranche-sur-Saône within the national distribution
# ============================================================================

villefranche_code = "69264"

comparison_features = [
    "retention_rate",
    "external_commuting_rate",
    "mean_external_commuting_distance_km",
    "share_external_commuters_0_5km",
    "share_external_commuters_5_10km",
    "share_external_commuters_10_20km",
    "share_external_commuters_20_40km",
    "share_external_commuters_over_40km",
    "share_external_commuters_under_20km",
    "share_external_commuters_5_20km",
]

# Compute percentile ranks across municipalities
percentile_ranks = (
    network_metrics[comparison_features]
    .rank(pct=True)
    .mul(100)
)

# Retrieve Villefranche values
villefranche_mask = (
    network_metrics["insee_code"] == villefranche_code
)

villefranche_values = (
    network_metrics
    .loc[villefranche_mask, comparison_features]
    .iloc[0]
)

villefranche_percentiles = (
    percentile_ranks
    .loc[villefranche_mask]
    .iloc[0]
)

# Build readable comparison table
villefranche_comparison = pd.DataFrame({
    "value": villefranche_values,
    "national_percentile": villefranche_percentiles,
})

villefranche_comparison